<a href="https://colab.research.google.com/github/uvindu827/J26-DS-350/blob/Indhi_Component2_CHL_Forecast/03_three_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ======================================================================
# STEP 31.9P-15 — FAIR THREE-MODEL COMPARISON
# RANDOM FOREST vs XGBOOST vs CNN-LSTM
# ======================================================================

import os
import json
import glob
import pandas as pd
import numpy as np

from pathlib import Path

# ======================================================================
# PROJECT PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

MODELS_DIR = PROJECT_DIR / "06_models"

RF_DIR = MODELS_DIR / "01_Random_Forest"
XGB_DIR = MODELS_DIR / "02_XGBoost"
CNN_DIR = MODELS_DIR / "03_CNN_LSTM"

COMPARISON_DIR = MODELS_DIR / "15_three_model_comparison"

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 70)
print("STEP 31.9P-15 — FAIR THREE-MODEL COMPARISON")
print("=" * 70)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nComparison directory:")
print(COMPARISON_DIR)


# ======================================================================
# HELPER FUNCTION — FIND CSV FILE
# ======================================================================

def find_csv(directory, keywords):
    """
    Search recursively for CSV files containing all requested keywords.
    """

    csv_files = list(directory.rglob("*.csv"))

    matches = []

    for file in csv_files:

        name = file.name.lower()

        if all(keyword.lower() in name for keyword in keywords):
            matches.append(file)

    return matches


# ======================================================================
# SEARCH FOR FINAL TEST METRIC FILES
# ======================================================================

print("\n" + "=" * 70)
print("SEARCHING FOR FINAL TEST METRICS")
print("=" * 70)


rf_candidates = find_csv(
    RF_DIR,
    ["test", "metric"]
)

xgb_candidates = find_csv(
    XGB_DIR,
    ["test", "metric"]
)

cnn_candidates = find_csv(
    CNN_DIR,
    ["test", "metric"]
)


print("\nRandom Forest metric candidates:")
for f in rf_candidates:
    print(" -", f)

print("\nXGBoost metric candidates:")
for f in xgb_candidates:
    print(" -", f)

print("\nCNN-LSTM metric candidates:")
for f in cnn_candidates:
    print(" -", f)


# ======================================================================
# SELECT FINAL METRIC FILES
# ======================================================================

def choose_final_metric_file(candidates, preferred_names):

    # First try exact preferred filenames
    for preferred in preferred_names:

        for file in candidates:

            if file.name.lower() == preferred.lower():
                return file

    # Otherwise use the first candidate containing FINAL
    final_candidates = [
        f for f in candidates
        if "final" in f.name.lower()
    ]

    if len(final_candidates) > 0:
        return final_candidates[0]

    if len(candidates) > 0:
        return candidates[0]

    return None


rf_metrics_file = choose_final_metric_file(
    rf_candidates,
    [
        "RANDOM_FOREST_FINAL_TEST_METRICS.csv",
        "RF_FINAL_TEST_METRICS.csv",
        "random_forest_final_test_metrics.csv"
    ]
)

xgb_metrics_file = choose_final_metric_file(
    xgb_candidates,
    [
        "XGBoost_FINAL_TEST_METRICS.csv"
    ]
)

cnn_metrics_file = choose_final_metric_file(
    cnn_candidates,
    [
        "CNN_LSTM_FINAL_TEST_METRICS.csv"
    ]
)


print("\n" + "=" * 70)
print("SELECTED FINAL TEST METRIC FILES")
print("=" * 70)

print("\nRandom Forest:")
print(rf_metrics_file)

print("\nXGBoost:")
print(xgb_metrics_file)

print("\nCNN-LSTM:")
print(cnn_metrics_file)


# ======================================================================
# SAFETY CHECK
# ======================================================================

if rf_metrics_file is None:
    raise FileNotFoundError(
        "Random Forest final test metric CSV could not be located."
    )

if xgb_metrics_file is None:
    raise FileNotFoundError(
        "XGBoost final test metric CSV could not be located."
    )

if cnn_metrics_file is None:
    raise FileNotFoundError(
        "CNN-LSTM final test metric CSV could not be located."
    )


print("\n✓ All three final metric files located.")


# ======================================================================
# LOAD METRICS
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL TEST METRICS")
print("=" * 70)


rf_metrics = pd.read_csv(rf_metrics_file)
xgb_metrics = pd.read_csv(xgb_metrics_file)
cnn_metrics = pd.read_csv(cnn_metrics_file)


print("\nRandom Forest metrics:")
print(rf_metrics.to_string(index=False))

print("\nXGBoost metrics:")
print(xgb_metrics.to_string(index=False))

print("\nCNN-LSTM metrics:")
print(cnn_metrics.to_string(index=False))


# ======================================================================
# STANDARDISE METRIC TABLE
# ======================================================================

def standardise_metrics(df, model_name):

    result = df.copy()

    result["model"] = model_name

    return result


rf_standard = standardise_metrics(
    rf_metrics,
    "Random Forest"
)

xgb_standard = standardise_metrics(
    xgb_metrics,
    "XGBoost"
)

cnn_standard = standardise_metrics(
    cnn_metrics,
    "CNN-LSTM"
)


comparison_raw = pd.concat(
    [
        rf_standard,
        xgb_standard,
        cnn_standard
    ],
    ignore_index=True
)


# ======================================================================
# SAVE RAW COMPARISON
# ======================================================================

raw_output = (
    COMPARISON_DIR /
    "three_model_final_test_metrics_raw.csv"
)

comparison_raw.to_csv(
    raw_output,
    index=False
)

print("\nRaw comparison saved:")
print(raw_output)


# ======================================================================
# CREATE CLEAN COMPARISON TABLE
# ======================================================================

print("\n" + "=" * 70)
print("CREATING FAIR COMPARISON TABLE")
print("=" * 70)


# The following function attempts to extract metrics regardless
# of whether the metric file is stored vertically or horizontally.

def extract_metric(df, possible_names):

    # --------------------------------------------------------------
    # Case 1 — metric/value format
    # --------------------------------------------------------------

    lower_columns = {
        str(c).lower(): c
        for c in df.columns
    }

    metric_column = None
    value_column = None

    for name in ["metric", "measure", "name"]:
        if name in lower_columns:
            metric_column = lower_columns[name]
            break

    for name in ["value", "score", "result"]:
        if name in lower_columns:
            value_column = lower_columns[name]
            break

    if metric_column is not None and value_column is not None:

        metrics_lower = (
            df[metric_column]
            .astype(str)
            .str.lower()
            .str.replace(" ", "_")
        )

        for possible in possible_names:

            possible_lower = (
                possible.lower()
                .replace(" ", "_")
            )

            mask = metrics_lower == possible_lower

            if mask.any():

                value = pd.to_numeric(
                    df.loc[mask, value_column],
                    errors="coerce"
                ).iloc[0]

                return value

    # --------------------------------------------------------------
    # Case 2 — metric stored directly as a column
    # --------------------------------------------------------------

    for column in df.columns:

        normalised = (
            str(column)
            .lower()
            .replace(" ", "_")
        )

        for possible in possible_names:

            possible_lower = (
                possible.lower()
                .replace(" ", "_")
            )

            if normalised == possible_lower:

                value = pd.to_numeric(
                    df[column],
                    errors="coerce"
                )

                if len(value) > 0:
                    return value.iloc[0]

    return np.nan


# ======================================================================
# EXTRACT CORE METRICS
# ======================================================================

models = {
    "Random Forest": rf_metrics,
    "XGBoost": xgb_metrics,
    "CNN-LSTM": cnn_metrics
}


comparison_rows = []


for model_name, df in models.items():

    row = {

        "model": model_name,

        # Original CHL
        "original_MAE": extract_metric(
            df,
            ["MAE", "original_MAE", "original_chl_MAE"]
        ),

        "original_RMSE": extract_metric(
            df,
            ["RMSE", "original_RMSE", "original_chl_RMSE"]
        ),

        "original_R2": extract_metric(
            df,
            ["R2", "R²", "original_R2", "original_chl_R2"]
        ),

        "original_Bias": extract_metric(
            df,
            ["Bias", "Mean Bias", "original_Bias"]
        ),

        # Log1p CHL
        "log1p_MAE": extract_metric(
            df,
            ["log_MAE", "log1p_MAE", "log1p_chl_MAE"]
        ),

        "log1p_RMSE": extract_metric(
            df,
            ["log_RMSE", "log1p_RMSE", "log1p_chl_RMSE"]
        ),

        "log1p_R2": extract_metric(
            df,
            ["log_R2", "log1p_R2", "log1p_chl_R2"]
        ),

        "log1p_Bias": extract_metric(
            df,
            ["log_Bias", "log1p_Bias", "log1p_chl_Bias"]
        )
    }

    comparison_rows.append(row)


comparison = pd.DataFrame(
    comparison_rows
)


# ======================================================================
# DISPLAY CORE COMPARISON
# ======================================================================

print("\n" + "=" * 70)
print("CORE THREE-MODEL TEST COMPARISON")
print("=" * 70)

print(
    comparison.to_string(index=False)
)


# ======================================================================
# SAVE CORE COMPARISON
# ======================================================================

core_output = (
    COMPARISON_DIR /
    "three_model_core_test_comparison.csv"
)

comparison.to_csv(
    core_output,
    index=False
)

print("\nCore comparison saved:")
print(core_output)


# ======================================================================
# BEST MODEL BY EACH METRIC
# ======================================================================

print("\n" + "=" * 70)
print("METRIC-BY-METRIC WINNERS")
print("=" * 70)


metric_directions = {

    "original_MAE": "lower",
    "original_RMSE": "lower",
    "original_R2": "higher",
    "original_Bias": "absolute_lower",

    "log1p_MAE": "lower",
    "log1p_RMSE": "lower",
    "log1p_R2": "higher",
    "log1p_Bias": "absolute_lower"
}


winner_rows = []


for metric, direction in metric_directions.items():

    values = pd.to_numeric(
        comparison[metric],
        errors="coerce"
    )

    if values.notna().sum() == 0:
        winner = "Not available"

    else:

        if direction == "lower":
            index = values.idxmin()

        elif direction == "higher":
            index = values.idxmax()

        elif direction == "absolute_lower":
            index = values.abs().idxmin()

        winner = comparison.loc[index, "model"]

    winner_rows.append(
        {
            "metric": metric,
            "best_model": winner
        }
    )


winners = pd.DataFrame(winner_rows)

print(
    winners.to_string(index=False)
)


winner_output = (
    COMPARISON_DIR /
    "three_model_metric_winners.csv"
)

winners.to_csv(
    winner_output,
    index=False
)


# ======================================================================
# MODEL RANKING
# ======================================================================

print("\n" + "=" * 70)
print("MODEL RANKING")
print("=" * 70)


# Rank only metrics that are actually available.
# Lower MAE/RMSE = better.
# Higher R2 = better.
# Smaller absolute bias = better.

ranking = comparison[
    [
        "model",
        "original_MAE",
        "original_RMSE",
        "original_R2",
        "original_Bias",
        "log1p_MAE",
        "log1p_RMSE",
        "log1p_R2",
        "log1p_Bias"
    ]
].copy()


for metric in [
    "original_MAE",
    "original_RMSE",
    "log1p_MAE",
    "log1p_RMSE"
]:

    ranking[metric + "_rank"] = (
        ranking[metric]
        .rank(
            ascending=True,
            method="min"
        )
    )


for metric in [
    "original_R2",
    "log1p_R2"
]:

    ranking[metric + "_rank"] = (
        ranking[metric]
        .rank(
            ascending=False,
            method="min"
        )
    )


for metric in [
    "original_Bias",
    "log1p_Bias"
]:

    ranking[metric + "_rank"] = (
        ranking[metric]
        .abs()
        .rank(
            ascending=True,
            method="min"
        )
    )


rank_columns = [
    c for c in ranking.columns
    if c.endswith("_rank")
]


ranking["available_metric_count"] = (
    ranking[rank_columns]
    .notna()
    .sum(axis=1)
)

ranking["mean_rank"] = (
    ranking[rank_columns]
    .mean(axis=1)
)


ranking = ranking.sort_values(
    "mean_rank"
)


print(
    ranking[
        [
            "model",
            "available_metric_count",
            "mean_rank"
        ]
        + rank_columns
    ].to_string(index=False)
)


ranking_output = (
    COMPARISON_DIR /
    "three_model_ranking.csv"
)

ranking.to_csv(
    ranking_output,
    index=False
)


# ======================================================================
# FINAL INTERPRETATION — DATA DRIVEN
# ======================================================================

print("\n" + "=" * 70)
print("THREE-MODEL COMPARISON INTERPRETATION")
print("=" * 70)


if len(ranking) > 0:

    print(
        "\nRanking based on the available core test metrics:"
    )

    for position, (_, row) in enumerate(
        ranking.iterrows(),
        start=1
    ):

        print(
            f"{position}. {row['model']} "
            f"(mean metric rank: {row['mean_rank']:.3f})"
        )


print(
    "\nIMPORTANT:"
)

print(
    "This ranking is a statistical summary of the available "
    "test metrics only."
)

print(
    "It does NOT automatically mean that the top-ranked model "
    "is the scientifically best model."
)

print(
    "Extreme-CHL behaviour, prediction range, temporal stability, "
    "and the project's forecasting objective must also be considered."
)


# ======================================================================
# COMPARISON SUMMARY JSON
# ======================================================================

summary = {

    "comparison_stage":
        "31.9P-15",

    "models": [
        "Random Forest",
        "XGBoost",
        "CNN-LSTM"
    ],

    "training_performed":
        False,

    "test_data_used_for_new_training":
        False,

    "comparison_basis": [
        "Original CHL MAE",
        "Original CHL RMSE",
        "Original CHL R2",
        "Original CHL bias",
        "Log1p CHL MAE",
        "Log1p CHL RMSE",
        "Log1p CHL R2",
        "Log1p CHL bias"
    ],

    "ranking_warning":
        "Core statistical metrics should be considered together "
        "with extreme-event behaviour and scientific objectives."
}


summary_file = (
    COMPARISON_DIR /
    "three_model_comparison_summary.json"
)


with open(
    summary_file,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ======================================================================
# COMPLETE
# ======================================================================

print("\n" + "=" * 70)
print("STEP 31.9P-15 COMPLETE")
print("=" * 70)

print(
    "\n✓ Random Forest final results loaded."
)

print(
    "✓ XGBoost final results loaded."
)

print(
    "✓ CNN-LSTM final results loaded."
)

print(
    "✓ Core test metrics compared."
)

print(
    "✓ Metric-by-metric winners calculated."
)

print(
    "✓ Model ranking calculated."
)

print(
    "✓ Comparison files saved."
)

print(
    "\nComparison directory:"
)

print(
    COMPARISON_DIR
)

print(
    "\nIMPORTANT:"
)

print(
    "No model was retrained."
)

print(
    "No hyperparameters were changed."
)

print(
    "No new model selection was performed using test data."
)

STEP 31.9P-15 — FAIR THREE-MODEL COMPARISON

Project directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Comparison directory:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/15_three_model_comparison

SEARCHING FOR FINAL TEST METRICS

Random Forest metric candidates:

XGBoost metric candidates:
 - /content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/evaluation/XGBoost_FINAL_TEST_METRICS.csv

CNN-LSTM metric candidates:
 - /content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/evaluation/CNN_LSTM_FINAL_TEST_METRICS.csv

SELECTED FINAL TEST METRIC FILES

Random Forest:
None

XGBoost:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/02_XGBoost/evaluation/XGBoost_FINAL_TEST_METRICS.csv

CNN-LSTM:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/03_CNN_LSTM/evaluation/CNN_LSTM_FINAL_TEST_METRICS.csv


FileNotFoundError: Random Forest final test metric CSV could not be located.

In [2]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

print("=" * 70)
print("RANDOM FOREST FILE SEARCH")
print("=" * 70)

# Search the entire project, not only the RF folder
all_files = list(PROJECT_DIR.rglob("*"))

print("\nFiles/folders containing 'random' or 'rf':")
print("-" * 70)

matches = []

for file in all_files:
    name = file.name.lower()

    if (
        "random" in name
        or "random_forest" in name
        or name.startswith("rf")
    ):
        matches.append(file)

if matches:
    for file in matches:
        print(file)
else:
    print("No matching Random Forest files found.")


print("\n" + "=" * 70)
print("ALL CSV FILES CONTAINING TEST / METRIC / SUMMARY")
print("=" * 70)

csv_matches = []

for file in all_files:

    if file.is_file() and file.suffix.lower() == ".csv":

        name = file.name.lower()

        if (
            "test" in name
            or "metric" in name
            or "summary" in name
        ):
            csv_matches.append(file)

if csv_matches:
    for file in csv_matches:
        print(file)
else:
    print("No matching CSV files found.")


print("\n" + "=" * 70)
print("RANDOM FOREST DIRECTORY CHECK")
print("=" * 70)

rf_dirs = [
    p for p in all_files
    if p.is_dir()
    and (
        "random" in p.name.lower()
        or "random_forest" in p.name.lower()
        or p.name.lower() == "01_random_forest"
    )
]

for directory in rf_dirs:
    print("\nDirectory:")
    print(directory)

    try:
        children = list(directory.iterdir())

        print("Contents:")

        for child in children:
            print("  ", child.name)

    except Exception as e:
        print("Could not read:", e)

RANDOM FOREST FILE SEARCH

Files/folders containing 'random' or 'rf':
----------------------------------------------------------------------
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_chlorophyll_model.joblib
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_training_configuration.csv
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_validation_predictions.csv
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_final_validation_summary.csv
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_final_validation_report.txt
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/01_Random_Forest_CHL_Forecasting.ipynb
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Fores

In [3]:
# ======================================================================
# STEP 31.9P-15A
# RANDOM FOREST FINAL TEST EVALUATION
# ======================================================================
#
# Purpose:
#   Evaluate the ALREADY TRAINED Random Forest model on the untouched
#   final test dataset.
#
# IMPORTANT:
#   - No retraining
#   - No hyperparameter tuning
#   - No model modification
#   - Test data used ONLY for final evaluation
# ======================================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    explained_variance_score,
    median_absolute_error
)

# ======================================================================
# PATHS
# ======================================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Indhi_Marine_Research_Data"
)

PREPROCESSING_DIR = (
    PROJECT_DIR / "05_model_preprocessing"
)

RF_DIR = (
    PROJECT_DIR / "06_models" / "01_Random_Forest"
)

RF_EVAL_DIR = RF_DIR / "evaluation"

RF_EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEST_FILE = (
    PREPROCESSING_DIR /
    "final_model_ready_test.csv"
)

FEATURE_FILE = (
    PREPROCESSING_DIR /
    "final_model_feature_columns.csv"
)

MODEL_FILE = (
    RF_DIR /
    "random_forest_chlorophyll_model.joblib"
)

print("=" * 70)
print("STEP 31.9P-15A")
print("RANDOM FOREST FINAL TEST EVALUATION")
print("=" * 70)

print("\nProject:")
print(PROJECT_DIR)

print("\nTest dataset:")
print(TEST_FILE)

print("\nFeature list:")
print(FEATURE_FILE)

print("\nRandom Forest model:")
print(MODEL_FILE)


# ======================================================================
# FILE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("REQUIRED FILE CHECK")
print("=" * 70)

for name, path in {
    "Test dataset": TEST_FILE,
    "Feature list": FEATURE_FILE,
    "Random Forest model": MODEL_FILE
}.items():

    print(f"\n{name}:")
    print(path)
    print("Exists:", path.exists())

    if not path.exists():
        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

print("\n✓ All required files found.")


# ======================================================================
# LOAD TEST DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING FINAL TEST DATA")
print("=" * 70)

test_df = pd.read_csv(
    TEST_FILE
)

feature_df = pd.read_csv(
    FEATURE_FILE
)

# Identify feature column
if "feature" in feature_df.columns:
    feature_columns = (
        feature_df["feature"]
        .dropna()
        .tolist()
    )

elif "feature_name" in feature_df.columns:
    feature_columns = (
        feature_df["feature_name"]
        .dropna()
        .tolist()
    )

else:
    feature_columns = feature_df.iloc[:, 0].dropna().tolist()


print("\nTest shape:")
print(test_df.shape)

print("\nNumber of selected features:")
print(len(feature_columns))

print("\nTarget:")
print("chlorophyll_a_log1p")

print("\nOriginal target:")
print("chlorophyll_a")


# ======================================================================
# FEATURE CHECK
# ======================================================================

print("\n" + "=" * 70)
print("FEATURE EXISTENCE CHECK")
print("=" * 70)

missing_features = [
    feature
    for feature in feature_columns
    if feature not in test_df.columns
]

print("\nMissing features:")
print(len(missing_features))

if missing_features:
    print(missing_features)
    raise ValueError(
        "Required model features are missing from test data."
    )

print("✓ All selected features are available.")


# ======================================================================
# BUILD TEST INPUTS
# ======================================================================

X_test = test_df[
    feature_columns
].copy()

y_test_log = test_df[
    "chlorophyll_a_log1p"
].copy()

y_test_original = test_df[
    "chlorophyll_a"
].copy()


print("\n" + "=" * 70)
print("TEST INPUT SHAPES")
print("=" * 70)

print("\nX_test:")
print(X_test.shape)

print("\ny_test_log:")
print(y_test_log.shape)

print("\ny_test_original:")
print(y_test_original.shape)


# ======================================================================
# SAFETY CHECK
# ======================================================================

print("\n" + "=" * 70)
print("TEST INPUT SAFETY CHECK")
print("=" * 70)

print(
    "\nPredictor missing values:",
    int(X_test.isna().sum().sum())
)

print(
    "Log target missing values:",
    int(y_test_log.isna().sum())
)

print(
    "Original target missing values:",
    int(y_test_original.isna().sum())
)

print(
    "Predictor infinite values:",
    int(np.isinf(X_test.to_numpy()).sum())
)

if (
    X_test.isna().sum().sum() > 0
    or y_test_log.isna().sum() > 0
    or y_test_original.isna().sum() > 0
    or np.isinf(X_test.to_numpy()).sum() > 0
):
    raise ValueError(
        "Test input safety check failed."
    )

print("\n✓ Test inputs are clean.")


# ======================================================================
# LOAD ALREADY TRAINED MODEL
# ======================================================================

print("\n" + "=" * 70)
print("LOADING ALREADY TRAINED RANDOM FOREST")
print("=" * 70)

rf_model = joblib.load(
    MODEL_FILE
)

print("\nModel type:")
print(type(rf_model))

print("\n✓ Existing Random Forest model loaded.")
print("✓ NO retraining performed.")


# ======================================================================
# GENERATE FINAL TEST PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("GENERATING FINAL TEST PREDICTIONS")
print("=" * 70)

print("\nTest rows:")
print(len(X_test))

y_pred_log = rf_model.predict(
    X_test
)

print("\n✓ Log1p predictions generated.")


# ======================================================================
# CONVERT BACK TO ORIGINAL CHL
# ======================================================================

print("\n" + "=" * 70)
print("CONVERTING PREDICTIONS TO ORIGINAL CHL")
print("=" * 70)

y_pred_original = np.expm1(
    y_pred_log
)

print("""
Random Forest output
        ↓
predicted log1p CHL
        ↓
expm1()
        ↓
predicted original CHL (mg/m³)
""")


# ======================================================================
# PREDICTION SAFETY
# ======================================================================

print("\n" + "=" * 70)
print("PREDICTION SANITY CHECK")
print("=" * 70)

print(
    "\nNaN predictions:",
    int(np.isnan(y_pred_log).sum())
)

print(
    "Infinite predictions:",
    int(np.isinf(y_pred_log).sum())
)

print(
    "NaN original predictions:",
    int(np.isnan(y_pred_original).sum())
)

print(
    "Infinite original predictions:",
    int(np.isinf(y_pred_original).sum())
)

if (
    np.isnan(y_pred_log).any()
    or np.isinf(y_pred_log).any()
    or np.isnan(y_pred_original).any()
    or np.isinf(y_pred_original).any()
):
    raise ValueError(
        "Prediction sanity check failed."
    )

print("\n✓ Predictions are valid.")


# ======================================================================
# PREDICTION RANGES
# ======================================================================

print("\n" + "=" * 70)
print("FINAL RANDOM FOREST TEST PREDICTION RANGE")
print("=" * 70)

print("\nActual log1p CHL:")
print("Minimum:", float(y_test_log.min()))
print("Maximum:", float(y_test_log.max()))

print("\nPredicted log1p CHL:")
print("Minimum:", float(y_pred_log.min()))
print("Maximum:", float(y_pred_log.max()))

print("\nActual original CHL:")
print(
    "Minimum:",
    float(y_test_original.min()),
    "mg/m³"
)

print(
    "Maximum:",
    float(y_test_original.max()),
    "mg/m³"
)

print("\nPredicted original CHL:")
print(
    "Minimum:",
    float(y_pred_original.min()),
    "mg/m³"
)

print(
    "Maximum:",
    float(y_pred_original.max()),
    "mg/m³"
)


# ======================================================================
# ORIGINAL CHL METRICS
# ======================================================================

original_mae = mean_absolute_error(
    y_test_original,
    y_pred_original
)

original_rmse = np.sqrt(
    mean_squared_error(
        y_test_original,
        y_pred_original
    )
)

original_r2 = r2_score(
    y_test_original,
    y_pred_original
)

original_median_ae = median_absolute_error(
    y_test_original,
    y_pred_original
)

original_bias = np.mean(
    y_pred_original -
    y_test_original
)

original_explained_variance = (
    explained_variance_score(
        y_test_original,
        y_pred_original
    )
)


# ======================================================================
# LOG1P METRICS
# ======================================================================

log_mae = mean_absolute_error(
    y_test_log,
    y_pred_log
)

log_rmse = np.sqrt(
    mean_squared_error(
        y_test_log,
        y_pred_log
    )
)

log_r2 = r2_score(
    y_test_log,
    y_pred_log
)

log_bias = np.mean(
    y_pred_log -
    y_test_log
)


# ======================================================================
# DISPLAY ORIGINAL SCALE PERFORMANCE
# ======================================================================

print("\n" + "=" * 70)
print("FINAL RANDOM FOREST TEST PERFORMANCE")
print("ORIGINAL CHL SCALE")
print("=" * 70)

print(
    f"MAE              : "
    f"{original_mae:.6f} mg/m³"
)

print(
    f"RMSE             : "
    f"{original_rmse:.6f} mg/m³"
)

print(
    f"R²               : "
    f"{original_r2:.6f}"
)

print(
    f"Median AE        : "
    f"{original_median_ae:.6f} mg/m³"
)

print(
    f"Mean Bias        : "
    f"{original_bias:.6f} mg/m³"
)

print(
    f"Explained Var.   : "
    f"{original_explained_variance:.6f}"
)


# ======================================================================
# DISPLAY LOG SCALE PERFORMANCE
# ======================================================================

print("\n" + "=" * 70)
print("FINAL RANDOM FOREST TEST PERFORMANCE")
print("LOG1P CHL SCALE")
print("=" * 70)

print(
    f"Log MAE     : "
    f"{log_mae:.6f}"
)

print(
    f"Log RMSE    : "
    f"{log_rmse:.6f}"
)

print(
    f"Log R²      : "
    f"{log_r2:.6f}"
)

print(
    f"Log Bias    : "
    f"{log_bias:.6f}"
)


# ======================================================================
# PREDICTION DIRECTION
# ======================================================================

error = (
    y_pred_original -
    y_test_original
)

over_count = int(
    (error > 0).sum()
)

under_count = int(
    (error < 0).sum()
)

exact_count = int(
    (error == 0).sum()
)

total = len(error)


print("\n" + "=" * 70)
print("FINAL RANDOM FOREST TEST PREDICTION DIRECTION")
print("=" * 70)

print(
    f"Overpredictions : "
    f"{over_count:,} "
    f"({over_count / total * 100:.2f}%)"
)

print(
    f"Underpredictions: "
    f"{under_count:,} "
    f"({under_count / total * 100:.2f}%)"
)

print(
    f"Exact predictions: "
    f"{exact_count:,} "
    f"({exact_count / total * 100:.2f}%)"
)


# ======================================================================
# SAVE TEST PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("SAVING RANDOM FOREST TEST PREDICTIONS")
print("=" * 70)

prediction_columns = {}

# Preserve metadata where available
for column in [
    "date",
    "target_date",
    "latitude",
    "longitude"
]:

    if column in test_df.columns:
        prediction_columns[column] = test_df[
            column
        ].values


prediction_columns[
    "actual_chlorophyll_a"
] = y_test_original.values

prediction_columns[
    "actual_chlorophyll_a_log1p"
] = y_test_log.values

prediction_columns[
    "predicted_chlorophyll_a_log1p"
] = y_pred_log

prediction_columns[
    "predicted_chlorophyll_a"
] = y_pred_original


rf_test_predictions = pd.DataFrame(
    prediction_columns
)


prediction_file = (
    RF_DIR /
    "random_forest_test_predictions.csv"
)

rf_test_predictions.to_csv(
    prediction_file,
    index=False
)

print("\nSaved:")
print(prediction_file)

print("\nShape:")
print(rf_test_predictions.shape)


# ======================================================================
# SAVE FINAL TEST METRICS
# ======================================================================

print("\n" + "=" * 70)
print("SAVING RANDOM FOREST FINAL TEST METRICS")
print("=" * 70)

metrics_rows = [

    {
        "scale": "original_chl",
        "MAE": original_mae,
        "RMSE": original_rmse,
        "R2": original_r2,
        "Median_AE": original_median_ae,
        "Bias": original_bias,
        "Explained_Variance":
            original_explained_variance
    },

    {
        "scale": "log1p_chl",
        "MAE": log_mae,
        "RMSE": log_rmse,
        "R2": log_r2,
        "Median_AE": np.nan,
        "Bias": log_bias,
        "Explained_Variance": np.nan
    }
]


rf_final_metrics = pd.DataFrame(
    metrics_rows
)

metrics_file = (
    RF_EVAL_DIR /
    "RANDOM_FOREST_FINAL_TEST_METRICS.csv"
)

rf_final_metrics.to_csv(
    metrics_file,
    index=False
)

print("\nSaved:")
print(metrics_file)

print("\nMetrics:")
print(
    rf_final_metrics.to_string(
        index=False
    )
)


# ======================================================================
# SAVE SUMMARY
# ======================================================================

summary = {

    "model":
        "Random Forest",

    "evaluation":
        "Final Test Evaluation",

    "retraining":
        False,

    "hyperparameter_tuning":
        False,

    "test_used_for_training":
        False,

    "test_used_for_final_evaluation":
        True,

    "number_of_features":
        len(feature_columns),

    "target":
        "chlorophyll_a_log1p",

    "original_target":
        "chlorophyll_a",

    "original_chl": {

        "MAE":
            float(original_mae),

        "RMSE":
            float(original_rmse),

        "R2":
            float(original_r2),

        "Median_AE":
            float(original_median_ae),

        "Bias":
            float(original_bias),

        "Explained_Variance":
            float(original_explained_variance)
    },

    "log1p_chl": {

        "MAE":
            float(log_mae),

        "RMSE":
            float(log_rmse),

        "R2":
            float(log_r2),

        "Bias":
            float(log_bias)
    },

    "prediction_range": {

        "actual_max":
            float(y_test_original.max()),

        "predicted_max":
            float(y_pred_original.max())
    }
}


summary_file = (
    RF_EVAL_DIR /
    "RANDOM_FOREST_FINAL_TEST_SUMMARY.json"
)

with open(
    summary_file,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# ======================================================================
# COMPLETE
# ======================================================================

print("\n" + "=" * 70)
print("STEP 31.9P-15A COMPLETE")
print("=" * 70)

print("\n✓ Existing Random Forest model loaded.")
print("✓ No retraining performed.")
print("✓ Final test dataset loaded.")
print("✓ Test predictions generated.")
print("✓ Log1p predictions converted using expm1().")
print("✓ Original CHL metrics calculated.")
print("✓ Log1p CHL metrics calculated.")
print("✓ Prediction range calculated.")
print("✓ Prediction direction calculated.")
print("✓ Test predictions saved.")
print("✓ Final RF test metrics saved.")
print("✓ Final RF test summary saved.")

print("\nFinal metrics:")
print(metrics_file)

print("\nTest predictions:")
print(prediction_file)

print("\nFinal summary:")
print(summary_file)

print("\n" + "=" * 70)
print("RANDOM FOREST FINAL TEST EVALUATION CLOSED")
print("=" * 70)

STEP 31.9P-15A
RANDOM FOREST FINAL TEST EVALUATION

Project:
/content/drive/MyDrive/Indhi_Marine_Research_Data

Test dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv

Random Forest model:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_chlorophyll_model.joblib

REQUIRED FILE CHECK

Test dataset:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_ready_test.csv
Exists: True

Feature list:
/content/drive/MyDrive/Indhi_Marine_Research_Data/05_model_preprocessing/final_model_feature_columns.csv
Exists: True

Random Forest model:
/content/drive/MyDrive/Indhi_Marine_Research_Data/06_models/01_Random_Forest/random_forest_chlorophyll_model.joblib
Exists: True

✓ All required files found.

LOADING FINAL TEST DATA

Test shape:
(157998, 75)

Nu